# 실험 E2 — Qwen3.5-9B + CoT (이미지)

**목적:** E1 위에 **Chain-of-Thought만** 얹어 추론 유도의 효과를 봅니다.

`E2 − E1` = CoT의 효과. E1과 모델·양자화·이미지가 모두 같고 **프롬프트와 출력 길이만** 다릅니다.

## E1과 달라지는 것 — 세 가지뿐

| | E1 | E2 |
|---|---|---|
| system | `ce.SYSTEM` (설명 금지) | **`ce.SYSTEM_COT`** (간단히 생각한 뒤 마지막 줄에 답) |
| 사용자 프롬프트 | `ce.make_prompt(row)` | **`ce.make_prompt(row, cot=True)`** |
| 출력 상한 | 24토큰 | **512토큰** |
| 파서 | `ce.parse_choice` (한 글자만 인정) | **`ce.parse_choice_cot`** |

질문과 선지 본문은 두 모드가 **완전히 동일**하고 마지막 지시문 한 줄만 교체됩니다.
`한 글자만 답하세요`를 남긴 채 CoT를 요구하면 지시가 충돌합니다.

**파서를 반드시 바꿔야 합니다.** `parse_choice`는 출력이 정확히 한 글자일 때만 인정하므로,
CoT 문장을 넣으면 672문항이 전부 INVALID가 됩니다.

## CoT가 손해일 수도 있습니다

B의 오답 상당수는 추론 실패가 아니라 **글자 오독**입니다. 읽기 실패에는 CoT가 도움이 되지 않고,
모델이 자기 오독을 논리로 정당화해 오히려 나빠질 수 있습니다. `E2 − E1`이 음수로 나와도 유효한 결과입니다.

비용도 큽니다. 출력이 24 → 512토큰이라 **추론 시간이 크게 늘어납니다.**

In [ ]:
# Resolve shared code and local assets from the repository root.
import os as _project_os
import sys as _project_sys
from pathlib import Path as _ProjectPath
_project_root = next((p for p in [_ProjectPath.cwd(), *_ProjectPath.cwd().parents]
                      if (p / 'src' / 'ocr_engine.py').is_file()), None)
if _project_root is not None:
    _project_os.chdir(_project_root)
    if str(_project_root / 'src') not in _project_sys.path:
        _project_sys.path.insert(0, str(_project_root / 'src'))
# End repository path setup.

import gc, json, os, time
from pathlib import Path

import torch
import pandas as pd
from tqdm.auto import tqdm

import common_eval as ce

VLM_MODEL_DIR = Path("downloads/models/Qwen3.5-9B")
OUTPUT_DIR = Path("output/exp_E2_qwen35_9b_cot")
MAX_NEW_TOKENS = 512        # CoT 추론 과정을 담을 길이
VLM_4BIT = True            # 9B bf16은 18.0 GiB라 16 GiB GPU에 올라가지 않습니다
ENABLE_THINKING = False    # Qwen3.5 자체 thinking 모드. CoT는 프롬프트로만 유도합니다
SMOKE_TEST = False         # True면 앞 8문항만

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
for key in ("HF_HUB_OFFLINE", "TRANSFORMERS_OFFLINE", "HF_DATASETS_OFFLINE"):
    os.environ[key] = "1"

assert torch.cuda.is_available(), "이 실험 설정은 CUDA GPU를 사용합니다."
assert (VLM_MODEL_DIR / "config.json").is_file(), f"로컬 모델을 준비하세요: {VLM_MODEL_DIR}"
shards = sorted(VLM_MODEL_DIR.glob("*.safetensors"))
index = VLM_MODEL_DIR / "model.safetensors.index.json"
if index.is_file():
    need = sorted(set(json.loads(index.read_text())["weight_map"].values()))
    miss = [f for f in need if not (VLM_MODEL_DIR / f).is_file()]
    assert not miss, f"가중치 shard {len(miss)}/{len(need)}개 누락. 다운로드를 끝내세요: {miss[:2]}"
assert shards, f"가중치(.safetensors)가 없습니다: {VLM_MODEL_DIR}"

DEVICE = "cuda:0"
DTYPE = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16
torch.manual_seed(42)
ce._self_test()
print("GPU:", torch.cuda.get_device_name(0), "/ dtype:", DTYPE)
print("shard", len(shards), "개 / 4bit:", VLM_4BIT)
print("이미지 긴 변 제한:", ce.IMAGE_MAX_EDGE, "px (실험 B/C와 동일)")

## 검증 분할

In [ ]:
val_df = ce.load_val()
if SMOKE_TEST:
    print("⚠ SMOKE_TEST: 공식 비교 결과로 사용 금지")
    val_df = val_df.head(8).copy()
rows = val_df.drop(columns=["answer"]).to_dict("records")   # 정답은 입력에서 제거
print("추론 대상:", len(rows))

## 프롬프트 (CoT)
마지막 줄 형식을 `정답: x`로 고정합니다. 형식을 고정하지 않으면 파서가 추론 과정에 등장하는
선지 언급을 최종 답으로 잘못 잡습니다.

In [ ]:
def build_messages(row, image):
    text = ce.make_prompt(row, cot=True)
    return [{"role": "system", "content": [{"type": "text", "text": ce.SYSTEM_COT}]},
            {"role": "user", "content": [{"type": "image", "image": image},
                                         {"type": "text", "text": text}]}]

print(build_messages(rows[0], None)[1]["content"][1]["text"])
print()
print("파서 동작 확인:", ce.parse_choice_cot("간판을 보면 ...\n정답: c"))

## 모델 로드
E1과 동일한 설정입니다.

In [ ]:
from transformers import AutoProcessor, Qwen3_5ForConditionalGeneration

def load_vlm():
    proc = AutoProcessor.from_pretrained(str(VLM_MODEL_DIR), local_files_only=True)
    kwargs = dict(local_files_only=True, dtype=DTYPE, device_map={"": 0})
    if VLM_4BIT:
        from transformers import BitsAndBytesConfig
        kwargs["quantization_config"] = BitsAndBytesConfig(
            load_in_4bit=True, bnb_4bit_quant_type="nf4",
            bnb_4bit_use_double_quant=True, bnb_4bit_compute_dtype=DTYPE)
    model = Qwen3_5ForConditionalGeneration.from_pretrained(str(VLM_MODEL_DIR), **kwargs)
    return model.eval(), proc

@torch.inference_mode()
def generate_text(model, proc, inputs, limit):
    inputs = inputs.to(model.device)
    prefix_len = inputs["input_ids"].shape[-1]
    output = model.generate(**inputs, max_new_tokens=limit, do_sample=False, use_cache=True)
    new = output[0, prefix_len:]
    eos = model.generation_config.eos_token_id
    eos_ids = [] if eos is None else ([eos] if isinstance(eos, int) else list(eos))
    truncated = len(new) >= limit and (not len(new) or int(new[-1]) not in eos_ids)
    return proc.decode(new, skip_special_tokens=True).strip(), bool(truncated), int(len(new))

## 추론
E1보다 오래 걸립니다. 출력 토큰이 21배이기 때문입니다.

In [ ]:
model, proc = load_vlm()
torch.cuda.reset_peak_memory_stats()
results = []
try:
    with (OUTPUT_DIR / "predictions.jsonl").open("w", encoding="utf-8") as log:
        for row in tqdm(rows, desc="E2: 9B + CoT"):
            torch.cuda.synchronize(); start = time.perf_counter()
            im = ce.open_image(row)
            inputs = proc.apply_chat_template(
                build_messages(row, im), add_generation_prompt=True, tokenize=True,
                return_dict=True, return_tensors="pt", enable_thinking=ENABLE_THINKING)
            raw, capped, n_tok = generate_text(model, proc, inputs, MAX_NEW_TOKENS)
            torch.cuda.synchronize(); elapsed = time.perf_counter() - start
            item = dict(id=row["id"], raw_output=raw, pred=ce.parse_choice_cot(raw),
                        answer_truncated=capped, answer_tokens=n_tok, seconds=elapsed,
                        image_width=im.width, image_height=im.height)
            results.append(item)
            log.write(json.dumps(item, ensure_ascii=False) + "\n"); log.flush()
            del inputs, im
    peak = torch.cuda.max_memory_allocated() / 1024**3
finally:
    del model, proc
    gc.collect(); torch.cuda.empty_cache()
print(f"완료 / GPU peak {peak:.2f} GiB")

## 분석
**`invalid_rate`를 먼저 보세요.** 높으면 모델이 `정답: x` 형식을 지키지 않은 것이고,
그러면 정확도는 CoT의 효과가 아니라 형식 실패를 측정한 값이 됩니다.
`answer_truncated_rate`가 높으면 512토큰 안에 결론을 못 낸 것이므로 상한을 올려야 합니다.

In [ ]:
pred_df = val_df.merge(pd.DataFrame(results), on="id", validate="one_to_one", sort=False)
assert pred_df.id.tolist() == val_df.id.tolist()
pred_df["correct"] = pred_df.pred == pred_df.answer

score = ce.save_analysis(pred_df, "E2_qwen35_9b_cot", OUTPUT_DIR, extra=dict(
    vlm_model=str(VLM_MODEL_DIR), vlm_4bit=VLM_4BIT, image_max_edge=ce.IMAGE_MAX_EDGE,
    answer_max_new_tokens=MAX_NEW_TOKENS, enable_thinking=ENABLE_THINKING,
    peak_allocated_gib=peak, cot=True,
    answer_truncated_rate=float(pred_df.answer_truncated.mean()),
    mean_answer_tokens=float(pred_df.answer_tokens.mean()),
    uses_image=True, uses_ocr=False, smoke=SMOKE_TEST))

# 예측이 한쪽으로 쏠리면 모델이 읽지 않고 찍고 있다는 신호입니다.
print("\n예측 분포 (정답 분포와 비슷해야 정상)")
comp = pd.DataFrame({"정답": val_df.answer.value_counts().reindex(ce.CHOICES),
                     "예측": pred_df.pred.value_counts().reindex(ce.CHOICES)}).fillna(0).astype(int)
print(comp.to_string())
print(f"\nB(Qwen2.5-VL-3B 단독) = 0.9033 과 비교: {score['accuracy'] - 0.9033:+.4f}")
print("산출물:", OUTPUT_DIR.resolve())

print("\n=== CoT 진단 ===")
print(f"INVALID(형식 실패)  {score['invalid']:4d}건 ({score['invalid_rate']:.1%})")
print(f"512토큰 내 미완결   {int(pred_df.answer_truncated.sum()):4d}건 ({pred_df.answer_truncated.mean():.1%})")
print(f"평균 생성 토큰       {pred_df.answer_tokens.mean():.1f}")
print("\n출력 샘플 2건:")
for _, r in pred_df.head(2).iterrows():
    print(f"  [{r.answer} -> {r.pred}] {str(r.raw_output)[:300]}")